# Data Cleaning
This notebook is used for the variables prices, load and generation dataset exploration phase, in order to understand how to organize the cleaning scripts.

## 1. Prices data

In [18]:
# import libraries
import pandas as pd
from pathlib import Path
import warnings

# filter User and Filter Warnings for privacy matters
warnings.simplefilter(action="ignore", category=UserWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)

In [19]:
# define raw dataset path
raw_dataset_path = Path('../dataset/raw/entsoe')

zone = 'IT_NORD'

# read data
prices_data = pd.read_csv(raw_dataset_path / 'prices' / f'{zone}-prices.csv')

prices_data

,Unnamed: 0,0
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01
...,...,...
2972,2026-01-31 23:00:00+01:00,122.77
2973,2026-01-31 23:15:00+01:00,122.77
2974,2026-01-31 23:30:00+01:00,122.77
2975,2026-01-31 23:45:00+01:00,120.00


In [20]:
# rename columns
prices_data = prices_data.rename(
    columns={
        'Unnamed: 0': 'datetime', 
        '0': f'prices_{zone}'
    }
)

prices_data

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01
...,...,...
2972,2026-01-31 23:00:00+01:00,122.77
2973,2026-01-31 23:15:00+01:00,122.77
2974,2026-01-31 23:30:00+01:00,122.77
2975,2026-01-31 23:45:00+01:00,120.00


In [21]:
# fast check
print(f'''
----Rows x Cols:----
{prices_data.shape}\n
----First rows:----
{prices_data.head()}\n
----Last rows:----
{prices_data.tail()}
''')


----Rows x Cols:----
(2977, 2)

----First rows:----
                    datetime  prices_IT_NORD
0  2026-01-01 00:00:00+01:00          111.42
1  2026-01-01 00:15:00+01:00          110.58
2  2026-01-01 00:30:00+01:00          110.58
3  2026-01-01 00:45:00+01:00          107.42
4  2026-01-01 01:00:00+01:00          106.01

----Last rows:----
                       datetime  prices_IT_NORD
2972  2026-01-31 23:00:00+01:00          122.77
2973  2026-01-31 23:15:00+01:00          122.77
2974  2026-01-31 23:30:00+01:00          122.77
2975  2026-01-31 23:45:00+01:00          120.00
2976  2026-02-01 00:00:00+01:00          140.11



In [22]:
# check what to clean --> Null values?
prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   datetime        2977 non-null   str    
 1   prices_IT_NORD  2977 non-null   float64
dtypes: float64(1), str(1)
memory usage: 46.6 KB


In [23]:
print(f'''
----Date description----
{prices_data['datetime'].describe()}\n
----Prices description----
{prices_data[f'prices_{zone}'].describe()}\n
''')


----Date description----
count                          2977
unique                         2977
top       2026-01-01 00:00:00+01:00
freq                              1
Name: datetime, dtype: object

----Prices description----
count    2977.000000
mean      133.805909
std        24.277683
min        85.260000
25%       117.250000
50%       129.460000
75%       148.780000
max       220.000000
Name: prices_IT_NORD, dtype: float64




In [24]:
# convert datetime
prices_data['datetime'] = pd.to_datetime(
    prices_data['datetime']
)

prices_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype                    
---  ------          --------------  -----                    
 0   datetime        2977 non-null   datetime64[us, UTC+01:00]
 1   prices_IT_NORD  2977 non-null   float64                  
dtypes: datetime64[us, UTC+01:00](1), float64(1)
memory usage: 46.6 KB


In [25]:
prices_data['datetime'].diff().value_counts()

datetime
0 days 00:15:00    2976
Name: count, dtype: int64

In [37]:
prices_data['datetime'].min()

Timestamp('2026-01-01 00:00:00+0100', tz='UTC+01:00')

In [38]:
prices_data['datetime'].max()

Timestamp('2026-02-01 00:00:00+0100', tz='UTC+01:00')

In [29]:
prices_data['datetime'].duplicated().sum()

np.int64(0)

In [39]:
# check timezone
print(prices_data['datetime'].dt.tz)

UTC+01:00


In [41]:
# check number of observations per day
daily_counts = (
    prices_data
    .set_index('datetime')
    .resample('D')
    .size()
)

daily_counts.value_counts().sort_index()

1      1
96    31
Name: count, dtype: int64

In [33]:
# function to get average hourly prices with different MTU data
def to_hourly(df, datetime_col='datetime', value_cols=None):
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = (
        df
        .set_index(datetime_col)
        .resample('h')[value_cols]
        .mean()
        .reset_index()
    )
    return df

In [35]:
# function call
prices_hourly = to_hourly(
    prices_data,
    datetime_col='datetime',
    value_cols=[f'prices_{zone}']
)

In [45]:
# Number of observations per hour
hourly_counts = (
    prices_data
    .set_index('datetime')
    .resample('h')['prices_IT_NORD']
    .count()
)

hourly_counts.value_counts().sort_index()

prices_IT_NORD
1      1
4    744
Name: count, dtype: int64

In [36]:
prices_hourly

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,110.0000
1,2026-01-01 01:00:00+01:00,109.0000
2,2026-01-01 02:00:00+01:00,108.1875
3,2026-01-01 03:00:00+01:00,105.1000
4,2026-01-01 04:00:00+01:00,98.9200
...,...,...
740,2026-01-31 20:00:00+01:00,142.1000
741,2026-01-31 21:00:00+01:00,129.7400
742,2026-01-31 22:00:00+01:00,125.5050
743,2026-01-31 23:00:00+01:00,122.0775


In [43]:
prices_hourly.info()

<class 'pandas.DataFrame'>
RangeIndex: 745 entries, 0 to 744
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype                    
---  ------          --------------  -----                    
 0   datetime        745 non-null    datetime64[us, UTC+01:00]
 1   prices_IT_NORD  745 non-null    float64                  
dtypes: datetime64[us, UTC+01:00](1), float64(1)
memory usage: 11.8 KB


In [44]:
prices_data[
    prices_data['datetime'].between(
        '2026-01-01 00:00',
        '2026-01-01 01:00'
    )
]

,datetime,prices_IT_NORD
0,2026-01-01 00:00:00+01:00,111.42
1,2026-01-01 00:15:00+01:00,110.58
2,2026-01-01 00:30:00+01:00,110.58
3,2026-01-01 00:45:00+01:00,107.42
4,2026-01-01 01:00:00+01:00,106.01


In [47]:
# export clean dataset
output_path = Path('../dataset/clean/prices')

# create folder if non-existent
output_path.mkdir(parents=True, exist_ok=True)

# save in CSV format
prices_hourly.to_csv(output_path / f'prices_{zone}.csv', index=False)

## 2. Load data

In [1]:
import pandas as pd
from pathlib import Path

raw_dataset_path = Path('../dataset/raw/entsoe')
zone = 'IT_NORD'

In [3]:
# load load_data
load_data = pd.read_csv(raw_dataset_path / 'load' / f'{zone}-load.csv')

load_data

,Unnamed: 0,Actual Load
0,2026-01-01 00:00:00+01:00,12425.0
1,2026-01-01 00:15:00+01:00,12315.0
2,2026-01-01 00:30:00+01:00,12200.0
3,2026-01-01 00:45:00+01:00,12088.0
4,2026-01-01 01:00:00+01:00,12017.0
...,...,...
2971,2026-01-31 22:45:00+01:00,15846.0
2972,2026-01-31 23:00:00+01:00,15843.0
2973,2026-01-31 23:15:00+01:00,15444.0
2974,2026-01-31 23:30:00+01:00,15153.0


In [4]:
# rename columns
load_data = load_data.rename(
    columns={
        'Unnamed: 0': 'datetime', 
        'Actual Load': f'Load_{zone}'
    }
)

load_data

,datetime,Load_IT_NORD
0,2026-01-01 00:00:00+01:00,12425.0
1,2026-01-01 00:15:00+01:00,12315.0
2,2026-01-01 00:30:00+01:00,12200.0
3,2026-01-01 00:45:00+01:00,12088.0
4,2026-01-01 01:00:00+01:00,12017.0
...,...,...
2971,2026-01-31 22:45:00+01:00,15846.0
2972,2026-01-31 23:00:00+01:00,15843.0
2973,2026-01-31 23:15:00+01:00,15444.0
2974,2026-01-31 23:30:00+01:00,15153.0


In [5]:
load_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2976 entries, 0 to 2975
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   datetime      2976 non-null   str    
 1   Load_IT_NORD  2976 non-null   float64
dtypes: float64(1), str(1)
memory usage: 46.6 KB


In [6]:
load_data.describe()

,Load_IT_NORD
count,2976.000000
mean,19575.933468
std,4874.779255
min,10271.000000
25%,15726.750000
50%,18426.000000
75%,24743.750000
max,28937.000000


In [8]:
# convert datetime
load_data['datetime'] = pd.to_datetime(
    load_data['datetime']
)

load_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2976 entries, 0 to 2975
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype                    
---  ------        --------------  -----                    
 0   datetime      2976 non-null   datetime64[us, UTC+01:00]
 1   Load_IT_NORD  2976 non-null   float64                  
dtypes: datetime64[us, UTC+01:00](1), float64(1)
memory usage: 46.6 KB


In [ ]:
# function to get average hourly prices with different MTU data
def to_hourly(df, datetime_col='datetime', value_cols=None):
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = (
        df
        .set_index(datetime_col)
        .resample('h')[value_cols]
        .mean()
        .reset_index()
    )
    return df

# function call
load_hourly = to_hourly(
    load_data,
    datetime_col='datetime',
    value_cols=[f'Load_{zone}']
)

In [11]:
load_hourly

,datetime,Load_IT_NORD
0,2026-01-01 00:00:00+01:00,12257.00
1,2026-01-01 01:00:00+01:00,11872.75
2,2026-01-01 02:00:00+01:00,11493.25
3,2026-01-01 03:00:00+01:00,11146.75
4,2026-01-01 04:00:00+01:00,10981.25
...,...,...
739,2026-01-31 19:00:00+01:00,20142.75
740,2026-01-31 20:00:00+01:00,18954.50
741,2026-01-31 21:00:00+01:00,17601.00
742,2026-01-31 22:00:00+01:00,16406.50


## 3. Generation data

In [12]:
import pandas as pd
from pathlib import Path

raw_dataset_path = Path('../dataset/raw/entsoe')
zone = 'IT_NORD'

In [13]:
generation_data = pd.read_csv(raw_dataset_path / 'generation' / f'{zone}-generation.csv')

generation_data

,Unnamed: 0,Biomass,Energy storage,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Wind Onshore
0,2026-01-01 00:00:00+01:00,141.0,NaN,9.0,9011.0,21.0,3.0,NaN,1067.0,68.0,1442.0,0.0,7.0
1,2026-01-01 00:15:00+01:00,136.0,NaN,9.0,8880.0,21.0,3.0,NaN,1043.0,69.0,1439.0,0.0,8.0
2,2026-01-01 00:30:00+01:00,138.0,NaN,9.0,9004.0,21.0,3.0,0.0,1042.0,71.0,1433.0,0.0,8.0
3,2026-01-01 00:45:00+01:00,136.0,NaN,9.0,8601.0,21.0,3.0,0.0,1041.0,69.0,1436.0,0.0,8.0
4,2026-01-01 01:00:00+01:00,137.0,5.0,9.0,8798.0,21.0,3.0,2.0,968.0,61.0,1434.0,0.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971,2026-01-31 22:45:00+01:00,124.0,NaN,10.0,7251.0,15.0,0.0,NaN,1013.0,74.0,1489.0,0.0,62.0
2972,2026-01-31 23:00:00+01:00,126.0,0.0,9.0,7268.0,15.0,0.0,NaN,986.0,73.0,1491.0,0.0,61.0
2973,2026-01-31 23:15:00+01:00,123.0,NaN,9.0,6808.0,15.0,0.0,NaN,986.0,73.0,1495.0,0.0,60.0
2974,2026-01-31 23:30:00+01:00,123.0,NaN,9.0,6431.0,15.0,0.0,NaN,987.0,74.0,1497.0,0.0,59.0


In [14]:
# rename columns
generation_data = generation_data.rename(
    columns={
        'Unnamed: 0': 'datetime'
    }
)

generation_data

,datetime,Biomass,Energy storage,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Wind Onshore
0,2026-01-01 00:00:00+01:00,141.0,NaN,9.0,9011.0,21.0,3.0,NaN,1067.0,68.0,1442.0,0.0,7.0
1,2026-01-01 00:15:00+01:00,136.0,NaN,9.0,8880.0,21.0,3.0,NaN,1043.0,69.0,1439.0,0.0,8.0
2,2026-01-01 00:30:00+01:00,138.0,NaN,9.0,9004.0,21.0,3.0,0.0,1042.0,71.0,1433.0,0.0,8.0
3,2026-01-01 00:45:00+01:00,136.0,NaN,9.0,8601.0,21.0,3.0,0.0,1041.0,69.0,1436.0,0.0,8.0
4,2026-01-01 01:00:00+01:00,137.0,5.0,9.0,8798.0,21.0,3.0,2.0,968.0,61.0,1434.0,0.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2971,2026-01-31 22:45:00+01:00,124.0,NaN,10.0,7251.0,15.0,0.0,NaN,1013.0,74.0,1489.0,0.0,62.0
2972,2026-01-31 23:00:00+01:00,126.0,0.0,9.0,7268.0,15.0,0.0,NaN,986.0,73.0,1491.0,0.0,61.0
2973,2026-01-31 23:15:00+01:00,123.0,NaN,9.0,6808.0,15.0,0.0,NaN,986.0,73.0,1495.0,0.0,60.0
2974,2026-01-31 23:30:00+01:00,123.0,NaN,9.0,6431.0,15.0,0.0,NaN,987.0,74.0,1497.0,0.0,59.0


In [15]:
generation_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2976 entries, 0 to 2975
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   datetime                         2976 non-null   str    
 1   Biomass                          2976 non-null   float64
 2   Energy storage                   1438 non-null   float64
 3   Fossil Coal-derived gas          2976 non-null   float64
 4   Fossil Gas                       2976 non-null   float64
 5   Fossil Hard coal                 2976 non-null   float64
 6   Fossil Oil                       2976 non-null   float64
 7   Hydro Pumped Storage             1329 non-null   float64
 8   Hydro Run-of-river and poundage  2976 non-null   float64
 9   Hydro Water Reservoir            2976 non-null   float64
 10  Other                            2976 non-null   float64
 11  Solar                            2976 non-null   float64
 12  Wind Onshore                   

In [16]:
generation_data.describe()

,Biomass,Energy storage,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Wind Onshore
count,2976.000000,1438.000000,2976.000000,2976.000000,2976.000000,2976.000000,1329.000000,2976.000000,2976.000000,2976.000000,2976.000000,2976.000000
mean,122.904906,228.483310,7.403226,11413.613239,29.532594,0.209677,302.868322,1274.712030,245.785618,1457.877016,543.654234,60.567540
std,13.771046,239.834322,1.954450,3397.824483,8.850938,0.734998,356.601322,318.280064,204.754248,38.338207,974.639563,44.688721
min,75.000000,0.000000,0.000000,4603.000000,15.000000,0.000000,0.000000,779.000000,39.000000,1347.000000,0.000000,1.000000
25%,111.000000,14.000000,6.000000,8434.750000,21.000000,0.000000,49.000000,1001.000000,80.000000,1432.000000,0.000000,18.000000
50%,129.000000,133.500000,7.500000,10891.000000,30.000000,0.000000,162.000000,1209.500000,174.000000,1454.000000,0.000000,52.000000
75%,135.000000,416.750000,9.000000,14893.000000,39.000000,0.000000,434.000000,1500.250000,365.250000,1482.000000,743.000000,107.000000
max,143.000000,736.000000,13.000000,17153.000000,41.000000,5.000000,2157.000000,2295.000000,1113.000000,1623.000000,4234.000000,136.000000


In [18]:
# convert datetime
generation_data['datetime'] = pd.to_datetime(
    generation_data['datetime']
)

generation_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 2976 entries, 0 to 2975
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype                    
---  ------                           --------------  -----                    
 0   datetime                         2976 non-null   datetime64[us, UTC+01:00]
 1   Biomass                          2976 non-null   float64                  
 2   Energy storage                   1438 non-null   float64                  
 3   Fossil Coal-derived gas          2976 non-null   float64                  
 4   Fossil Gas                       2976 non-null   float64                  
 5   Fossil Hard coal                 2976 non-null   float64                  
 6   Fossil Oil                       2976 non-null   float64                  
 7   Hydro Pumped Storage             1329 non-null   float64                  
 8   Hydro Run-of-river and poundage  2976 non-null   float64                  
 9   Hydro Water Reservo

In [22]:
gen = generation_data.columns[1:]
gen

Index(['Biomass', 'Energy storage', 'Fossil Coal-derived gas', 'Fossil Gas',
       'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage',
       'Hydro Run-of-river and poundage', 'Hydro Water Reservoir', 'Other',
       'Solar', 'Wind Onshore'],
      dtype='str')

In [28]:
for col in ['Energy storage', 'Hydro Pumped Storage']:
    print(f'\n--- {col} ---')
    print(generation_data[col].isna().value_counts())
    print(generation_data.loc[generation_data[col].isna(), 'datetime'].min())
    print(generation_data.loc[generation_data[col].isna(), 'datetime'].max())

    print(generation_data[col].head())
    print(generation_data[col].tail())


--- Energy storage ---
Energy storage
True     1538
False    1438
Name: count, dtype: int64
2026-01-01 00:00:00+01:00
2026-01-31 23:30:00+01:00
0    NaN
1    NaN
2    NaN
3    NaN
4    5.0
Name: Energy storage, dtype: float64
2971    NaN
2972    0.0
2973    NaN
2974    NaN
2975    9.0
Name: Energy storage, dtype: float64

--- Hydro Pumped Storage ---
Hydro Pumped Storage
True     1647
False    1329
Name: count, dtype: int64
2026-01-01 00:00:00+01:00
2026-01-31 23:45:00+01:00
0    NaN
1    NaN
2    0.0
3    0.0
4    2.0
Name: Hydro Pumped Storage, dtype: float64
2971   NaN
2972   NaN
2973   NaN
2974   NaN
2975   NaN
Name: Hydro Pumped Storage, dtype: float64


In [29]:
generation_data[['Energy storage', 'Hydro Pumped Storage']].describe()

,Energy storage,Hydro Pumped Storage
count,1438.000000,1329.000000
mean,228.483310,302.868322
std,239.834322,356.601322
min,0.000000,0.000000
25%,14.000000,49.000000
50%,133.500000,162.000000
75%,416.750000,434.000000
max,736.000000,2157.000000


In [31]:
for col in ['Energy storage', 'Hydro Pumped Storage']:
    print(f'\n{col}')
    print(generation_data[col].dropna().head(20).to_string(index=False))


Energy storage
  5.0
  9.0
 64.0
  0.0
  5.0
  5.0
 60.0
620.0
640.0
221.0
 17.0
  0.0
 43.0
454.0
322.0
273.0
276.0
289.0
482.0
417.0

Hydro Pumped Storage
0.0
0.0
2.0
0.0
0.0
0.0
0.0
0.0
0.0
2.0
0.0
0.0
0.0
0.0
2.0
0.0
4.0
3.0
0.0
1.0


In [33]:
for col in ['Energy storage', 'Hydro Pumped Storage']:

    missing_by_hour = (
        generation_data
        .set_index('datetime')[col]
        .isna()
        .resample('h')
        .sum()
    )

    print(f'\n--- {col} ---')
    print(missing_by_hour.value_counts().sort_index())


--- Energy storage ---
Energy storage
0    213
1    108
2     87
3     88
4    248
Name: count, dtype: int64

--- Hydro Pumped Storage ---
Hydro Pumped Storage
0    285
1     25
2     24
3     66
4    344
Name: count, dtype: int64


In [23]:
# function to get average hourly prices with different MTU data
def to_hourly(df, datetime_col='datetime', value_cols=None):
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col])
    df = (
        df
        .set_index(datetime_col)
        .resample('h')[value_cols]
        .mean()
        .reset_index()
    )
    return df

# function call
generation_hourly = to_hourly(
    generation_data,
    datetime_col='datetime', 
    value_cols=gen
)

generation_hourly

,datetime,Biomass,Energy storage,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Solar,Wind Onshore
0,2026-01-01 00:00:00+01:00,137.75,NaN,9.0,8874.00,21.00,3.0,0.00,1048.25,69.25,1437.50,0.0,7.75
1,2026-01-01 01:00:00+01:00,137.00,5.000000,9.0,8983.75,21.00,3.0,2.00,947.50,59.00,1432.25,0.0,9.00
2,2026-01-01 02:00:00+01:00,137.75,NaN,9.0,8893.50,21.00,3.0,0.00,938.25,58.00,1431.50,0.0,13.25
3,2026-01-01 03:00:00+01:00,136.75,NaN,8.0,8298.00,21.00,3.0,NaN,916.00,58.00,1432.25,0.0,37.00
4,2026-01-01 04:00:00+01:00,136.50,36.500000,8.0,7475.50,24.50,2.5,NaN,892.00,57.50,1439.00,0.0,44.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
739,2026-01-31 19:00:00+01:00,138.75,171.250000,10.0,10835.00,21.00,0.0,623.25,1493.25,540.00,1544.75,0.0,77.25
740,2026-01-31 20:00:00+01:00,137.50,1.000000,10.0,10593.25,21.00,0.0,244.50,1275.25,241.75,1532.00,0.0,68.50
741,2026-01-31 21:00:00+01:00,122.50,1.000000,10.0,9553.25,20.25,0.0,88.75,1081.25,114.25,1519.25,0.0,59.00
742,2026-01-31 22:00:00+01:00,123.00,45.666667,10.0,7663.00,15.00,0.0,2.00,1028.75,93.25,1496.25,0.0,60.75
